In [ ]:
import mysql.connector
import pandas as pd
import datetime as dt
import pytz
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import entropy
import seaborn as sns

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [ ]:
#Connection to  MySQL server, parameters must be filled with correct information otherwise it won't work
conn = mysql.connector.connect(
    host = 'must_be_inserted',
    user = 'must_be_inserted',
    password = 'must_be_inserted',
    database = 'test'
)
cursor = conn.cursor()

# Preparing all addresses

In [ ]:
#Extracting addresses with assigned timezone
sql = """SELECT eh.entity, eh.address, sd.timezone, sd.offsset FROM entities_heur as eh
    LEFT JOIN scraped_data as sd ON eh.entity = sd.entity
    LEFT JOIN exchanges_detection as ed on sd.id = ed.id
    WHERE sd.address is not null 
        and sd.location is not null 
        and sd.num_txs > 0 
        and sd.timezone is not null 
        and ed.arkham_entity is null 
        and (ed.arkham_label is null or ed.arkham_label = "Miner");"""
cursor.execute(sql)
data = cursor.fetchall()
len(data)

In [ ]:
data = pd.DataFrame(data, columns = ["entity", "address", "timezone", "offset"])
data.head()

In [ ]:
data = data[~data["offset"].isna()]
len(data)

In [ ]:
#Transforming address to list and deleting duplicates
addresses = data["address"]
addresses = addresses.drop_duplicates()
addresses = addresses.tolist()

# Getting transaction data

In [ ]:
#Getting all outgoing transaction to addresses from list
placeholders = ','.join(['%s'] * len(addresses))

query = f"""SELECT DISTINCT ti.txid, ti.input_order, ti.address, bd.mempool_entry_time, bd.fee, bd.num_inputs, bd.num_outputs, ti.value FROM tx_inputs AS ti
LEFT JOIN blockchain_data AS bd on ti.txid = bd.txid
WHERE address in ({placeholders});"""

cursor.execute(query, addresses)
database = cursor.fetchall()
database = pd.DataFrame(database, columns = ["txid", "input_order", "address", "mempool_entry_time", "fee", "num_inputs", "num_outputs", "value"])
database.head()

In [ ]:
#Merging transaction data to addresses to obtain entity identification number
db_final = database.merge(
    data,
    how = "left",
    on = "address"
)

In [ ]:
#Creating help variable with adjusted time to UTC +0:00
db_final["time_adjusted"] = db_final.apply(
    lambda r: r["mempool_entry_time"] + pd.Timedelta(hours = r["offset"])- pd.Timedelta(hours = 1),
    axis = 1
)
db_final = db_final.drop_duplicates(subset = ["txid"])
db_final.head()

In [ ]:
len(db_final)

In [ ]:
#General  graph of transaction activity tranformed to UTC +0:00
counts, bins, patches = plt.hist(db_final["time_adjusted"].dt.hour, bins=np.arange(25), 
                                 rwidth=0.8, color='skyblue', edgecolor='black')

plt.xticks(np.arange(24) + 0.5, labels=range(24))

plt.title("Distribution of Transactions by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Number of transactions")

plt.show()

# Creating regions

In [ ]:
#Summary of entities and transactions numbers per each timezone
summary = (
    db_final.groupby("offset")
        .agg(
            num_of_entities=("entity", "nunique"),
            num_of_txs=("txid", "count")
        )
        .reset_index()
)
summary["avg_num_of_txs_per_entity"] = round(summary["num_of_txs"]/summary["num_of_entities"],2)
summary = summary.sort_values("offset")
summary["entities%"] = round(summary["num_of_entities"]/sum(summary["num_of_entities"])*100,2)
summary

In [ ]:
sum(summary["num_of_entities"])

In [ ]:
#Mapping and grouping data by defined regions
region_mapping = {
    # America
    -8.0: 'America', -7.0: 'America', -6.0: 'America', -5.0: 'America', -4.0: 'America', -3.0: 'America',
    
    # Euro-Africa
    0.0: 'Euro_Africa', 1.0: 'Euro_Africa', 2.0: 'Euro_Africa', 3.0: 'Euro_Africa',
    
    # Central Asia / India / Middle East
    3.5: 'Central_Asia', 4.0: 'Central_Asia', 4.5: 'Central_Asia', 5.0: 'Central_Asia', 
    5.5: 'Central_Asia', 5.75: 'Central_Asia', 6.0: 'Central_Asia', 6.5: 'Central_Asia',
    
    # East Asia / Pacific
    7.0: 'East_Asia_Pac', 8.0: 'East_Asia_Pac', 9.0: 'East_Asia_Pac', 10.0: 'East_Asia_Pac', 10.5: 'East_Asia_Pac', 11.0: 'East_Asia_Pac', 
    12.0: 'East_Asia_Pac', 13.0: 'East_Asia_Pac', -10.0: 'East_Asia_Pac', -9.0: 'East_Asia_Pac'
}

db_final['region'] = db_final['offset'].map(region_mapping)

In [ ]:
#Summary of entities and transactions numbers per each region
summary = (
    db_final.groupby("region")
        .agg(
            num_of_entities=("entity", "nunique"),
            num_of_txs=("txid", "count")
        )
        .reset_index()
)
summary["avg_num_of_txs_per_entity"] = round(summary["num_of_txs"]/summary["num_of_entities"],2)
summary = summary.sort_values("region")
summary["entities%"] = round(summary["num_of_entities"]/sum(summary["num_of_entities"])*100,2)
summary

# Creating final dataset

In [ ]:
#Extracting only hour of transaction from dataset
db_final["hour"] = pd.to_datetime(db_final["mempool_entry_time"]).dt.hour

In [ ]:
#Aggregation of transaction into entities
pivot = (
    db_final.pivot_table(
        index=["entity", "region"],
        columns="hour",
        values="txid",
        aggfunc="count",
        fill_value=0
    )
)

In [ ]:
pivot.head()

In [ ]:
table = pivot.reset_index()
table.columns.name = None
table.head()

In [ ]:
cols_to_sum = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
table["total"] = table[cols_to_sum].sum(axis = 1)
table["log_total"] = table["total"].apply(lambda r: np.log(r))

#Bayesian normalization of data 
alpha = 1
table[cols_to_sum] = (table[cols_to_sum] + alpha).div(table["total"] + 24*alpha, axis = 0)

In [ ]:
#Finding the middle of the 5 hour minimum inactivity window and the % value of this window
hourly_data = table[cols_to_sum]
circular_data = pd.concat([hourly_data, hourly_data.iloc[:,:4]], axis = 1)
rolling_sums = circular_data.T.rolling(window=5).sum().T
hours = (rolling_sums.iloc[:, 4:].idxmin(axis=1).astype(int) - 2)%24
table["min_5h_activity"] = hours
table["min_5h_activity_sin"] = np.sin(hours*np.pi/12)
table["min_5h_activity_cos"] = np.cos(hours*np.pi/12)
table["min_5h_activity_%"] = rolling_sums.iloc[:, 4:].min(axis=1)

In [ ]:
table.head()

In [ ]:
table.to_csv("database_from_Bitcointalk.csv", index = False)

# Data without labels

In [ ]:
#Addresses which dont have timezone but has higher number of transaction than 0
sql = """SELECT eh.entity, eh.address FROM entities_heur as eh
    LEFT JOIN scraped_data as sd ON eh.entity = sd.entity
    LEFT JOIN exchanges_detection as ed on sd.id = ed.id
    WHERE sd.address is not null 
        and sd.location is null 
        and sd.num_txs > 0 
        and sd.timezone is null 
        and ed.arkham_entity is null 
        and (ed.arkham_label is null or ed.arkham_label = "Miner");"""
cursor.execute(sql)
data = cursor.fetchall()
len(data)

In [ ]:
data = pd.DataFrame(data, columns = ["entity", "address"])
data.head()

In [ ]:
addresses = data["address"]
addresses = addresses.drop_duplicates()
addresses = addresses.tolist()
addresses

In [ ]:
#Getting transactions data
placeholders = ','.join(['%s'] * len(addresses))

query = f"""SELECT DISTINCT ti.txid, ti.input_order, ti.address, bd.mempool_entry_time, bd.fee, bd.num_inputs, bd.num_outputs, ti.value FROM tx_inputs AS ti
LEFT JOIN blockchain_data AS bd on ti.txid = bd.txid
WHERE address in ({placeholders});"""

cursor.execute(query, addresses)
database = cursor.fetchall()
database = pd.DataFrame(database, columns = ["txid", "input_order", "address", "mempool_entry_time", "fee", "num_inputs", "num_outputs", "value"])
database.head()

In [ ]:
db_final = database.merge(
    data,
    how = "left",
    on = "address"
)

In [ ]:
db_final["hour"] = pd.to_datetime(db_final["mempool_entry_time"]).dt.hour
pivot = (
    db_final.pivot_table(
        index=["entity"],
        columns="hour",
        values="txid",
        aggfunc="count",
        fill_value=0
    )
)
table = pivot.reset_index()
table.columns.name = None
table.head()

In [ ]:
cols_to_sum = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
table["total"] = table[cols_to_sum].sum(axis = 1)
table["log_total"] = table["total"].apply(lambda r: np.log(r))

#Bayesian normalization of data 
alpha = 1
table[cols_to_sum] = (table[cols_to_sum] + alpha).div(table["total"] + 24*alpha, axis = 0)

#Finding the middle of the 5 hour minimum inactivity window and the % value of this window
hourly_data = table[cols_to_sum]
circular_data = pd.concat([hourly_data, hourly_data.iloc[:,:4]], axis = 1)
rolling_sums = circular_data.T.rolling(window=5).sum().T
hours = (rolling_sums.iloc[:, 4:].idxmin(axis=1).astype(int) - 2)%24
table["min_5h_activity"] = hours
table["min_5h_activity_sin"] = np.sin(hours*np.pi/12)
table["min_5h_activity_cos"] = np.cos(hours*np.pi/12)
table["min_5h_activity_%"] = rolling_sums.iloc[:, 4:].min(axis=1)

table.head()

In [ ]:
table.to_csv("database_from_Bitcointalk_not_labeled.csv", index = False)

In [ ]:
len(table)